<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    target_dir = f"{REPO_DIR}/work/notebooks"
    if os.path.basename(os.getcwd()) != "notebooks":
        os.chdir(target_dir)
print("Ready! Current Working Directory:", os.getcwd())

Ready! Current Working Directory: /content/flyrank-internship-assignment1/work/notebooks


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked Actions & Reason Codes
The model ranks/flags content for review based on predicted decay probability. It outputs a prioritized queue grouped by the following reason codes:  

Code A (High Priority Review): High decay probability (>0.70) + High impressions_90d. Action: Human review for immediate content refresh.

Code B (Monitor): Moderate decay probability (0.50 - 0.70). Action: Monitor trailing metrics; no immediate action required.

Code C (Low Priority): Low decay probability (<0.50). Action: Maintain as is.

In [3]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Load data and train model
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']
df_clean = df.dropna(subset=features + ['is_declining_label', 'client_id'])

X = df_clean[features]
y = df_clean['is_declining_label']
groups = df_clean['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

# Generate Queue and apply Reason Codes
playbook_queue = X.iloc[test_idx].copy()
playbook_queue['client_id'] = df_clean.iloc[test_idx]['client_id']
playbook_queue['decay_probability'] = rf.predict_proba(X.iloc[test_idx])[:, 1]

def assign_reason_code(row):
    if row['decay_probability'] > 0.70 and row['impressions_90d'] > 1000: return 'Code A (High Priority)'
    elif row['decay_probability'] >= 0.50: return 'Code B (Monitor)'
    else: return 'Code C (Low Priority)'

playbook_queue['action_code'] = playbook_queue.apply(assign_reason_code, axis=1)
playbook_queue = playbook_queue.sort_values(by='decay_probability', ascending=False)
display(playbook_queue[['client_id', 'decay_probability', 'action_code']].head())

,client_id,decay_probability,action_code
8719,client_8527a891e2,0.813173,Code A (High Priority)
18191,client_8527a891e2,0.812505,Code B (Monitor)
12651,client_8527a891e2,0.812505,Code B (Monitor)
14927,client_8527a891e2,0.812491,Code B (Monitor)
13469,client_8527a891e2,0.812491,Code A (High Priority)


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Decision-Support Only: This model provides decision-support for human reviewers. Because we are analyzing cross-sectional data without a controlled design, we can only state that we observed patterns in this dataset; the model's scores do not prove that updating a page causes traffic to return.
  
Data Limits: The model cannot differentiate between a page ranking at position zero and a page with missing data, as avg_position = 0 simply means "no data". Furthermore, because history depth differs wildly per client, global calendar windows cannot be universally applied.

In [4]:
# Proving the data limit: Count how many rows have the "no data" position trap
zero_position_count = len(df[df['avg_position'] == 0])
print(f"Data Limit Flag: There are {zero_position_count} rows where avg_position=0 (Meaning 'No Data', not Rank 0).")

Data Limit Flag: There are 1205 rows where avg_position=0 (Meaning 'No Data', not Rank 0).


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Content updates should NEVER be fully automated. A human must review the top of the ranked queue to filter out the model's blind spots.

Static Intent (No-Go): Glossary terms, historical records, or definition pages. Even if the model flags them due to age, human reviewers must skip them.

Missingness Injection (No-Go): Some content types naturally have missing keyword data or word counts. Reviewers must ensure a page isn't being flagged just because its specific format lacks standard text metrics.

In [5]:
# Proving the no-go list: Identifying items that naturally lack word counts
missing_word_count = df['word_count'].isna().sum()
print(f"Human Review Required: {missing_word_count} items are missing word counts and must not be blindly penalized.")

Human Review Required: 7699 items are missing word counts and must not be blindly penalized.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain Trigger 1: When a significant volume of new clients is onboarded to the data warehouse.

Retrain Trigger 2: If the out-of-sample Precision@50 on new data drops below the naive base rate (majority class).

In [6]:
# Calculate the baseline trigger for monitoring
base_rate = y.mean()
print(f"Monitoring Trigger: If out-of-sample Precision@50 drops below {base_rate:.2f} (the base rate), RETRAIN model.")

Monitoring Trigger: If out-of-sample Precision@50 drops below 0.57 (the base rate), RETRAIN model.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
import pandas as pd
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data & Prepare Target
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']
df = df.dropna(subset=features + ['is_declining_label', 'client_id'])

X = df[features]
y = df['is_declining_label']
groups = df['client_id']

# 2. Honest Grouped Split (Training the final playbook model)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# 3. Generate Ranked Queue for the Playbook
playbook_queue = X_test.copy()
playbook_queue['client_id'] = df.iloc[test_idx]['client_id']
playbook_queue['decay_probability'] = rf.predict_proba(X_test)[:, 1]

# 4. Apply Reason Codes
def assign_reason_code(row):
    if row['decay_probability'] > 0.70 and row['impressions_90d'] > 1000:
        return 'Code A (High Priority Review)'
    elif row['decay_probability'] >= 0.50:
        return 'Code B (Monitor)'
    else:
        return 'Code C (Low Priority)'

playbook_queue['action_code'] = playbook_queue.apply(assign_reason_code, axis=1)

# 5. Sort Queue and Export
playbook_queue = playbook_queue.sort_values(by='decay_probability', ascending=False)
os.makedirs('../outputs', exist_ok=True)
export_path = '../outputs/playbook_ranked_queue.csv'
playbook_queue.head(100).to_csv(export_path, index=False)

print(f"Success! Ranked playbook queue exported to: {export_path}")
display(playbook_queue[['client_id', 'decay_probability', 'action_code']].head())

Success! Ranked playbook queue exported to: ../outputs/playbook_ranked_queue.csv


,client_id,decay_probability,action_code
8719,client_8527a891e2,0.813173,Code A (High Priority Review)
18191,client_8527a891e2,0.812505,Code B (Monitor)
12651,client_8527a891e2,0.812505,Code B (Monitor)
14927,client_8527a891e2,0.812491,Code B (Monitor)
13469,client_8527a891e2,0.812491,Code A (High Priority Review)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.